# GVH Diagonal Cubic 0.3.2.7.3.7.2.5 — Generic Full-Field Total Kinetic Inverse and Complete Normal Constraint Density

**Auteur :** Charlemagne O Laurince

## Mission

Tenter de fermer \(R_{DD2}\) génériquement à partir de
\[
\mathcal L=\frac12V^TQ_{\rm total}V+J^TV+U
\]
et
\[
\mathcal C_\perp
=
\frac12(P-J)^TQ_{\rm total}^{-1}(P-J)-U.
\]

Ce notebook adopte une règle stricte : **aucun FULL-PASS ne sera déclaré si l'inverse générique full-field n'est pas effectivement reconstruite**.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:
from __future__ import annotations
import sympy as sp, json, sys, time
from pathlib import Path
print("GVH 0.3.2.7.3.7.2.5")
print("Python:",sys.version.split()[0])
print("SymPy:",sp.__version__)


GVH 0.3.2.7.3.7.2.5
Python: 3.13.5
SymPy: 1.14.0


## 1. Hessien cinétique générique local

Le Hessien ne dépend pas des gradients spatiaux \(D_is,D_iv_j,a_i^{(n)}\) : ceux-ci alimentent \(J\) et \(U\). On peut donc reconstruire \(Q_{\rm total}\) avec ces gradients mis à zéro sans perdre d'information cinétique.


In [2]:
c1,c2,c3,c4,s = sp.symbols("c1 c2 c3 c4 s", real=True)
v1,v2,v3 = sp.symbols("v1 v2 v3", real=True)
v = sp.Matrix([v1,v2,v3])

K11,K22,K33,K12,K13,K23,S,W1,W2,W3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3", real=True)
vel=[K11,K22,K33,K12,K13,K23,S,W1,W2,W3]

K=sp.Matrix([[K11,K12,K13],[K12,K22,K23],[K13,K23,K33]])
W=sp.Matrix([W1,W2,W3])

A=-S
B=W-K*v
C=-K*v
D=s*K

I1=sp.expand(A**2-B.dot(B)-C.dot(C)+sum(D[i,j]**2 for i in range(3) for j in range(3)))
theta=sp.expand(-A+sp.trace(D))
I3=sp.expand(A**2-2*B.dot(C)+sum(D[i,j]*D[j,i] for i in range(3) for j in range(3)))
alpha=sp.expand(s*A+v.dot(C))
beta=sp.expand(s*B+D.T*v)
a2=sp.expand(-alpha**2+beta.dot(beta))

Lu=sp.expand(-c1*I1-c2*theta**2-c3*I3+c4*a2)
Qu=sp.hessian(Lu,vel)

LEH=sp.expand(sum(K[i,j]**2 for i in range(3) for j in range(3))-sp.trace(K)**2)
QEH=sp.zeros(10,10)
QEH6=sp.hessian(LEH,vel[:6])
for i in range(6):
    for j in range(6):
        QEH[i,j]=QEH6[i,j]

Qtotal=sp.simplify(QEH+Qu)
assert Qtotal==Qtotal.T
print("Q_total generic symbolic shape =",Qtotal.shape)
print("Generic Hessian assembly: PASS")


Q_total generic symbolic shape = (10, 10)
Generic Hessian assembly: PASS


## 2. Décomposition bloc exacte

On écrit
\[
Q_{\rm total}
=
\begin{pmatrix}
A_6 & B\\
B^T & D_4
\end{pmatrix},
\]
où \(D_4\) porte \((S,W_i)\).

Si \(D_4\) et le complément de Schur
\[
S_6=A_6-BD_4^{-1}B^T
\]
sont inversibles, alors
\[
Q_{\rm total}^{-1}
=
\begin{pmatrix}
S_6^{-1} & -S_6^{-1}BD_4^{-1}\\
-D_4^{-1}B^TS_6^{-1} &
D_4^{-1}+D_4^{-1}B^TS_6^{-1}BD_4^{-1}
\end{pmatrix}.
\]


In [3]:
A6=Qtotal[:6,:6]
B64=Qtotal[:6,6:10]
D4=Qtotal[6:10,6:10]

detD4=sp.factor(D4.det())
D4inv=sp.simplify(D4.inv())
S6=sp.simplify(A6-B64*D4inv*B64.T)

print("det(D4) =",detD4)
print("D4 inverse explicit: PASS")
print("Schur complement S6 explicit shape =",S6.shape)


det(D4) = -16*(c1 + c4*s**2)**3*(c1 + c2 + c3 + c4*s**2)
D4 inverse explicit: PASS
Schur complement S6 explicit shape = (6, 6)


Le déterminant du bloc \(D_4\) donne directement des surfaces de dégénérescence :
\[
\det D_4
=
-16(c_1+c_4s^2)^3(c_1+c_2+c_3+c_4s^2).
\]


In [4]:
expected=-16*(c1+c4*s**2)**3*(c1+c2+c3+c4*s**2)
assert sp.expand(detD4-expected)==0
print("D4 determinant factorization: PASS")


D4 determinant factorization: PASS


## 3. Test de non-dégénérescence générique du complément de Schur

Au lieu de prétendre avoir une factorisation symbolique complète de \(\det S_6\), on vérifie une branche rationnelle exacte. Cela démontre l'existence d'une région non dégénérée, mais **pas encore** l'inverse symbolique générale de \(S_6\).


In [5]:
witness={
 c1:sp.Rational(2,5), c2:sp.Rational(1,7),
 c3:sp.Rational(-1,11), c4:sp.Rational(3,13),
 s:sp.Rational(5,4),
 v1:sp.Rational(1,5), v2:sp.Rational(-1,6), v3:sp.Rational(1,7),
}
Qw=sp.Matrix(Qtotal.subs(witness))
S6w=sp.Matrix(S6.subs(witness))
print("rank Q_total witness =",Qw.rank())
print("rank S6 witness =",S6w.rank())
assert Qw.rank()==10
assert S6w.rank()==6
print("Generic nondegenerate witness: PASS")


rank Q_total witness = 10
rank S6 witness = 6
Generic nondegenerate witness: PASS


## 4. Inverse bloc exacte au témoin

On vérifie que la formule de Schur reconstruit exactement \(Q^{-1}\) sur la branche rationnelle.


In [6]:
A6w=Qw[:6,:6]
B64w=Qw[:6,6:10]
D4w=Qw[6:10,6:10]
D4iw=D4w.inv()
S6iw=(A6w-B64w*D4iw*B64w.T).inv()

Qinv_block_w = sp.Matrix.vstack(
    sp.Matrix.hstack(S6iw, -S6iw*B64w*D4iw),
    sp.Matrix.hstack(-D4iw*B64w.T*S6iw,
                     D4iw+D4iw*B64w.T*S6iw*B64w*D4iw)
)
assert Qw*Qinv_block_w == sp.eye(10)
print("Exact Schur block inverse identity on witness: PASS")


Exact Schur block inverse identity on witness: PASS


## 5. Contrainte normale complète : formule générique conditionnelle

Dès que \(S_6^{-1}\) sera rendu symboliquement explicite, l'inverse full-field sera fermée et l'on aura
\[
\boxed{
\mathcal C_\perp
=
\frac12(P-J)^TQ_{\rm total}^{-1}(P-J)-U.
}
\]

Les objets \(J\) et \(U\) ont déjà été reconstruits dans 7.7.2.4. Le seul verrou restant est donc maintenant localisé dans
\[
\boxed{S_6^{-1}\text{ symbolique générique}.}
\]


In [7]:
P=sp.Matrix(sp.symbols("P0:10", real=True))
J=sp.Matrix(sp.symbols("J0:10", real=True))
U=sp.symbols("U_total", real=True)
Pi=P-J

Cperp_w=sp.expand(sp.Rational(1,2)*(Pi.T*Qinv_block_w*Pi)[0]-U)
print("Complete C_perp on exact witness branch: PASS")


Complete C_perp on exact witness branch: PASS


## 6. Audit de \(R_{DD2}\)

Le notebook obtient :
- \(Q_{\rm total}\) générique explicite ;
- \(D_4^{-1}\) générique explicite ;
- \(S_6\) générique explicite ;
- existence d'une branche avec \(\mathrm{rank}(S_6)=6\) ;
- inverse bloc exacte vérifiée au témoin.

Mais il ne publie pas encore \(S_6^{-1}\) en symbolique générique. Par conséquent :
\[
\boxed{R_{DD2}:\ \text{REDUCED, NOT YET PROVEN ZERO GENERICALLY}}
\]


In [8]:
GATES={
 "generic_Qtotal_explicit":True,
 "generic_D4_inverse_explicit":True,
 "generic_Schur_complement_explicit":True,
 "generic_nondegenerate_witness":True,
 "exact_block_inverse_on_witness":True,
 "complete_Cperp_on_witness":True,
 "full_Ci_explicit_generic":True,

 "generic_S6_inverse_explicit":False,
 "generic_full_field_Q_inverse_published":False,
 "full_Cperp_explicit_generic":False,
 "RDD2_computed":False,
 "hypersurface_algebra_closed":False,
}
for k,vv in GATES.items():
    print(k,":",vv)

FINAL_STATUS=(
 "PARTIAL-PASS-GENERIC-QTOTAL-D4-INVERSE-AND-SCHUR-COMPLEMENT-EXPLICIT_"
 "EXACT-BLOCK-INVERSE-ON-WITNESS_"
 "BLOCKED-GENERIC-S6-INVERSE-AND-GENERIC-C-PERP"
)
DISPERSION_READY=False

assert not GATES["RDD2_computed"]
assert DISPERSION_READY is False
print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


generic_Qtotal_explicit : True
generic_D4_inverse_explicit : True
generic_Schur_complement_explicit : True
generic_nondegenerate_witness : True
exact_block_inverse_on_witness : True
complete_Cperp_on_witness : True
full_Ci_explicit_generic : True
generic_S6_inverse_explicit : False
generic_full_field_Q_inverse_published : False
full_Cperp_explicit_generic : False
RDD2_computed : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-GENERIC-QTOTAL-D4-INVERSE-AND-SCHUR-COMPLEMENT-EXPLICIT_EXACT-BLOCK-INVERSE-ON-WITNESS_BLOCKED-GENERIC-S6-INVERSE-AND-GENERIC-C-PERP
DISPERSION_READY = False


## 7. Prochaine sous-étape

Le verrou est désormais suffisamment réduit pour justifier une sous-étape spécialisée :

### `0.3.2.7.3.7.2.6 — Analytic Schur-Sector Decomposition and Generic Six-Dimensional Inverse`

Objectifs :
1. exploiter la structure tensorielle de \(S_6\) ;
2. décomposer le secteur métrique symétrique en sous-espaces longitudinal/transverse par rapport à \(v_i\) ;
3. inverser analytiquement les petits blocs résultants ;
4. reconstruire \(S_6^{-1}\), puis \(Q_{\rm total}^{-1}\) ;
5. seulement alors passer `RDD2_computed=True`.

Le passage à 7.7.3 reste différé.


In [9]:
artifact={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.5",
 "final_status":FINAL_STATUS,
 "Qtotal_generic_explicit":True,
 "D4_inverse_generic_explicit":True,
 "Schur_S6_generic_explicit":True,
 "Schur_S6_inverse_generic_explicit":False,
 "RDD2_status":"REDUCED_NOT_YET_PROVEN_ZERO_GENERICALLY",
 "gates":GATES,
 "dispersion_ready":False,
 "next":"GVH_Diagonal_Cubic_0.3.2.7.3.7.2.6_Analytic_Schur_Sector_Decomposition_and_Generic_Six_Dimensional_Inverse.ipynb"
}
export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path.cwd()/"gvh_exports"
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.7.2.5_generic_schur_inverse_audit.json"
artifact_path.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",artifact_path)


Artifact: /home/oai/gvh_exports/gvh_0.3.2.7.3.7.2.5_generic_schur_inverse_audit.json


# Conclusion

7.7.2.5 réduit fortement le dernier verrou de \(R_{DD2}\), mais ne le ferme pas artificiellement.

\[
\boxed{
Q_{\rm total}
\rightarrow
D_4^{-1}
\rightarrow
S_6
}
\]
sont maintenant explicites génériquement.

L'étape manquante est précisément :
\[
\boxed{S_6^{-1}\text{ générique explicite}.}
\]

Verdict :
\[
\boxed{\text{PARTIAL PASS}}
\]
et
\[
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
